In [ ]:
import os
import numpy as np
import SimpleITK as sitk
import matplotlib.pyplot as plt

In [ ]:
moving = 'MD585'
fixed = 'Allen'
structure = 'IC'
atlas_path = "/net/birdstore/Active_Atlas_Data/data_root/atlas_data"
reg_path = "/net/birdstore/Active_Atlas_Data/data_root/brains_info/registration"

In [ ]:
fixed_image_path = os.path.join(reg_path, fixed, 'Allen_10.0x10.0x10.0um_sagittal.tif')
moving_image_path = os.path.join(reg_path, moving, 'MD585_10.0x10.0x10.0um_sagittal.tif')
affine_transform_path = os.path.join(reg_path, moving, 'MD585_Allen_10.0x10.0x10.0um.tfm')
origin_dir = os.path.join(atlas_path, moving, 'origin')        
masks_dir = os.path.join(atlas_path, moving, 'structure') 
output_overlay_path = os.path.join(reg_path, moving, 'test_overlay.tif')

In [ ]:
fixed_image = sitk.ReadImage(fixed_image_path, sitk.sitkFloat32)
moving_image = sitk.ReadImage(moving_image_path, sitk.sitkFloat32)
transform = sitk.ReadTransform(affine_transform_path)

In [ ]:
origins = sorted([f for f in os.listdir(origin_dir)])
masks = sorted([f for f in os.listdir(masks_dir)])

print(origins)
print(masks)

In [ ]:
registered_masks = []
start = 5
end = start + 10
for origin, mask in zip(origins[start:end], masks[start:end]):
    print(f'loading {origin=} {mask=}')
    mask_path = os.path.join(masks_dir, mask)
    origin_path = os.path.join(origin_dir, origin)
    origin = np.loadtxt(origin_path)
    mask_np = np.load(mask_path)
    mask_np[mask_np > 0] = 255
    mask_np = mask_np.astype(np.uint8)
    mask_np = np.swapaxes(mask_np, 0, 2)
    # Create SimpleITK image for mask
    mask_sitk = sitk.GetImageFromArray(mask_np)
    mask_sitk.SetOrigin(origin) # very important!!!!

    # Apply affine transform
    resampled_mask = sitk.Resample(
        mask_sitk,
        fixed_image,
        transform,
        sitk.sitkNearestNeighbor,   # Important for binary masks!
        0.0,
        sitk.sitkUInt8
    )
    registered_masks.append(resampled_mask)
print(f'Finished length of registered masks={len(registered_masks)}')

In [ ]:
combined_mask = sitk.Cast(sitk.Maximum(registered_masks[0], registered_masks[0]*0), sitk.sitkUInt8)
for m in registered_masks:
    combined_mask = sitk.Maximum(combined_mask, m)

# ----------------------------
# Overlay registered masks onto fixed volume
# ----------------------------
overlay = sitk.LabelOverlay(sitk.Cast(fixed_image, sitk.sitkUInt8), combined_mask)
sitk.WriteImage(overlay, output_overlay_path)

# ----------------------------
# Visualize a middle slice
# ----------------------------
mid_slice = fixed_image.GetSize()[2] // 2
fixed_np = sitk.GetArrayFromImage(fixed_image)[mid_slice]
mask_np = sitk.GetArrayFromImage(combined_mask)[mid_slice]

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(fixed_np, cmap='gray')
plt.imshow(mask_np, cmap='Reds', alpha=0.4)
plt.title(f"Overlay Slice {mid_slice}")
plt.axis('off')
plt.show()